[instructions & rubric](https://course.ccs.neu.edu/ds4400f26-higger/learn/puzzle_problems.html)

# Puzzle: Slice Crawl

There are 8 pizza shops near campus and you want to find the best one.
The catch: you learn about a shop only by tasting its slices.

**Your goal: taste as few slices as possible while still naming the best
shop with at least 95% confidence** (right in at least 95% of crawls).

Each slice gets a rating, but slices vary: the same shop can be great on
Tuesday and soggy on Friday. Each shop's ratings follow a bell curve around
that shop's typical rating, and some shops are more consistent than others.
You don't know any shop's typical rating or consistency.

`SliceCrawl` (in `slice_crawl.py`) runs one crawl. `crawl.taste(i)` buys a
slice from shop `i` and returns its rating. Write `estimate(crawl)`: taste
any shops you like, as many times as you like, then return the shop you
think is best.

- First, clear the bar: name the true best shop in at least 95% of crawls.
- Then, save pizza: among estimators that clear the bar, fewer slices
  (averaged over many crawls) is better.
- Practice on the
  [Slice Crawl site](https://matthigger.github.io/ml_puzzle/slice_crawl/#practice):
  it runs your `estimate()` on as many random crawls as you like and shows
  how it did.
- For fun, we'll race everyone's estimator head to head: fewest slices
  among those that clear 95% wins bragging rights (no effect on your grade).
  Post yours on the site's Leaderboard tab with class code `slice-n30b-xqvq-92e4`.

`evaluate(estimate)` runs your estimator on 200 random crawls and reports
how often it was right and how many slices it tasted on average (the same
`seed` always gives the same crawls). Anything on a crawl that starts with
an underscore (e.g. `crawl._mean`) is the hidden truth: peeking is
cheating. An example estimator:

In [2]:
from slice_crawl import SliceCrawl, evaluate

crawl = SliceCrawl.sample(seed=0)  # one random crawl of 8 shops
print(crawl.taste(3), crawl.taste(3), crawl.n_taste)


def taste_each_5(crawl):
    """ 5 slices from every shop, best average wins """
    avg = [sum(crawl.taste(i) for _ in range(5)) / 5
           for i in range(crawl.n_shop)]
    return avg.index(max(avg))


evaluate(taste_each_5);

5.5158722675697405 4.67064038868554 2
200 crawls: right 76.0% of the time (need 95%), 40.0 slices on average


In [ ]:
import random

from slice_crawl import SliceCrawl, evaluate
from slice_helpers import N0, THRESHOLD, SEED, summarize, prob_best, top_two_pick


def estimate(crawl):
    """Bayesian estimator: taste until the leader's P(best) >= THRESHOLD."""
    rng = random.Random(SEED)

    # step 1: taste N0 slices from every shop
    ratings = [[crawl.taste(i) for _ in range(N0)] for i in range(crawl.n_shop)]
    # steps 2-3: mean, standard error, df for every shop
    summaries = [summarize(r) for r in ratings]

    while True:
        # steps 4-5: P(best) for every shop; stop if the leader is sure enough
        p = prob_best(summaries, rng)
        leader = p.index(max(p))
        if p[leader] >= THRESHOLD:
            return leader

        # otherwise taste ~5% more slices (chosen by top_two_pick), then
        # re-check. Re-checking only every ~5% keeps it fast.
        for _ in range(max(1, crawl.n_taste // 20)):
            pick = top_two_pick(summaries, rng)
            ratings[pick].append(crawl.taste(pick))
            summaries[pick] = summarize(ratings[pick])


evaluate(estimate);